# 3. GARCH and Conditional Volatility

Financial returns exhibit **volatility clustering**: periods of high volatility tend to persist. The **GARCH(1,1)** model captures this:
$$r_t = \sigma_t \varepsilon_t, \quad \sigma_t^2 = \omega + \alpha r_{t-1}^2 + \beta \sigma_{t-1}^2$$

This notebook covers:
- Simulating a GARCH(1,1) process
- Fitting GARCH models with the `arch` library
- Comparing GARCH, EGARCH, and GJR-GARCH
- Volatility forecasting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from arch import arch_model

%matplotlib inline
np.random.seed(42)

## 3.1 Simulating GARCH(1,1)

We simulate returns from a GARCH(1,1) with $\omega=0.05$, $\alpha=0.10$, $\beta=0.85$.
Note that $\alpha+\beta=0.95 < 1$, ensuring stationarity.

In [ ]:
n = 1000
omega, alpha, beta = 0.05, 0.10, 0.85
sigma2 = np.zeros(n)
r = np.zeros(n)
sigma2[0] = omega / (1 - alpha - beta)

for t in range(1, n):
    sigma2[t] = omega + alpha * r[t-1]**2 + beta * sigma2[t-1]
    r[t] = np.sqrt(sigma2[t]) * np.random.normal()

print(f"Unconditional variance: {omega/(1-alpha-beta):.4f}")
print(f"Empirical variance:     {np.var(r):.4f}")

## 3.2 Visualizing Volatility Clustering

Notice how large returns (in absolute value) cluster together, and quiet periods also cluster.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8))
axes[0].plot(r, lw=0.5)
axes[0].set_title('Simulated Returns (GARCH(1,1))')
axes[1].plot(r**2, lw=0.5, color='orange')
axes[1].set_title('Squared Returns')
axes[2].plot(np.sqrt(sigma2), lw=0.5, color='red')
axes[2].set_title('Conditional Volatility $\\sigma_t$')
plt.tight_layout()
plt.show()

## 3.3 Fitting GARCH(1,1)

The `arch` library provides maximum likelihood estimation for GARCH-family models.

In [ ]:
am = arch_model(r, vol='Garch', p=1, q=1, dist='normal')
res = am.fit(disp='off')
print(res.summary())

## 3.4 Model Comparison: GARCH vs EGARCH vs GJR-GARCH

- **EGARCH**: models log-volatility, captures asymmetric effects
- **GJR-GARCH**: adds a leverage term for negative returns

Compare using AIC and BIC.

In [ ]:
models = {
    'GARCH(1,1)': arch_model(r, vol='Garch', p=1, q=1),
    'EGARCH(1,1)': arch_model(r, vol='EGARCH', p=1, q=1),
    'GJR-GARCH(1,1)': arch_model(r, vol='Garch', p=1, o=1, q=1),
}

for name, model in models.items():
    try:
        result = model.fit(disp='off')
        print(f"{name}: AIC={result.aic:.2f}, BIC={result.bic:.2f}")
    except Exception as e:
        print(f"{name}: failed ({e})")

## 3.5 Volatility Forecasting

GARCH models provide **forward-looking variance forecasts**, essential for risk management (VaR, Expected Shortfall).

In [ ]:
forecasts = res.forecast(horizon=10)
print("Variance forecast (next 10 periods):")
print(forecasts.variance.iloc[-1])

## Key Takeaways

- GARCH captures **volatility clustering** observed in financial returns
- The condition $\alpha + \beta < 1$ ensures **stationarity**
- **EGARCH** and **GJR-GARCH** capture asymmetric (leverage) effects
- GARCH forecasts are crucial for **risk management** applications